In [0]:
# Criclane — 02 Ingestion (long-running poller)
# Runs as the production ingestion job. One loop:
#   fast tick (default 5s):  /api/home  + live match pages (round-robin)
#   every 60s:               scorecards for live + recently finished matches
#   every 10 min:            schedule, series news, points tables
#   every 60 min:            ICC rankings (men/women x batting/bowling/all-rounder/teams)
# After every fetch cycle: bronze -> silver -> gold refresh (data lands in gold
# within seconds of it changing on Cricbuzz).
# Unchanged payloads are content-hash skipped, so bronze only grows when data changes.

import sys, time, json
from collections import deque

LIB_DIR = "/Workspace/Users/ritik69ritik@gmail.com/Cricbuzz ETL Pipeline"
if LIB_DIR not in sys.path:
    sys.path.insert(0, LIB_DIR)
import criclane_lib as cl

dbutils.widgets.text("run_seconds", "120", "Run duration in seconds (0 = run forever)")
dbutils.widgets.text("fast_interval", "5", "Fast tick seconds (live scores)")
dbutils.widgets.text("scorecard_interval", "60", "Scorecard refresh seconds")
dbutils.widgets.text("slow_interval", "600", "Schedule/news/points-table seconds")
dbutils.widgets.text("rankings_interval", "3600", "Rankings refresh seconds")
dbutils.widgets.text("max_live_pages_per_cycle", "4", "Live match pages fetched per tick")
dbutils.widgets.text("series_cap", "12", "Max series for news/points-table fetches")

P = {k: dbutils.widgets.get(k) for k in ["run_seconds", "fast_interval", "scorecard_interval",
     "slow_interval", "rankings_interval", "max_live_pages_per_cycle", "series_cap"]}
P = {k: float(v) for k, v in P.items()}
print("Criclane ingest parameters:", P)

In [0]:
# ---------------- Ingestion loop ----------------

end_at = time.time() + P["run_seconds"] if P["run_seconds"] > 0 else None
next_due = {"scorecard": 0.0, "slow": 0.0, "rankings": 0.0}
hashes = {}                       # (kind, id) -> content hash (skip unchanged payloads)
live_rr = deque()                 # round-robin of live match ids
consecutive_failures = 0
cycle = 0


def append(bronze, rows):
    cl.append_rows(spark, cl.bronze_table(bronze), rows, cl.BRONZE_SCHEMAS[bronze])


def ingest_home():
    """Fetch /api/home; write changed match payloads to bronze. Returns (live_ids, recent_done, series_map).""
    matches = cl.fetch_home_matches()
    rows, live, recent_done, series_map = [], [], [], {}
    for x in matches:
        info = (x.get("match") or {}).get("matchInfo") or {}
        mid = cl.to_int(info.get("matchId"))
        if mid is None:
            continue
        if info.get("seriesId"):
            series_map[cl.to_int(info.get("seriesId"))] = (info.get("seriesName") or "", cl.slugify(info.get("seriesName")))
        h = cl.content_hash(x)
        if hashes.get(("home", mid)) != h:
            rows.append({"match_id": mid, "series_id": cl.to_int(info.get("seriesId")),
                         "state": info.get("state"), "payload": json.dumps(x),
                         "payload_hash": h, "fetch_ts": cl.now_utc(), "processed": False})
            hashes[("home", mid)] = h
        st = info.get("state")
        if st == "In Progress":
            live.append(mid)
        elif st == "Complete" and cl.epoch_ms_to_ts(info.get("endDate")):
            end_ts = cl.epoch_ms_to_ts(info.get("endDate"))
            if (cl.now_utc() - end_ts).total_seconds() < 12 * 3600:
                recent_done.append(mid)
    if rows:
        append("home_matches", rows)
    return live, recent_done, series_map


print(f"Criclane poller starting — fast tick {P['fast_interval']:.0f}s, deadline {'never' if not end_at else end_at:.0f}")

while True:
    cycle += 1
    t0 = time.time()
    if end_at and t0 >= end_at:
        print(f"Run window elapsed after {cycle - 1} cycles. Exiting.")
        break

    stats = {}
    try:
        live, recent_done, series_map = ingest_home()
        consecutive_failures = 0

        # --- round-robin live match pages (miniscore + commentary) ---
        for mid in live:
            if mid not in live_rr:
                live_rr.append(mid)
        for mid in list(live_rr):
            if mid not in live:
                live_rr.remove(mid)
        for _ in range(min(int(P["max_live_pages_per_cycle"]), len(live_rr))):
            if not live_rr:
                break
            mid = live_rr.popleft()
            data = cl.fetch_match_live(mid)
            if data and (data.get("miniscore") or data.get("commentary")):
                h = cl.content_hash(data)
                if hashes.get(("live", mid)) != h:
                    append("match_miniscore", [{"match_id": mid, "payload": json.dumps(data),
                                                "payload_hash": h, "fetch_ts": cl.now_utc(),
                                                "processed": False}])
                    hashes[("live", mid)] = h
            if mid in live:
                live_rr.append(mid)

        # --- scorecards (live + recently finished) ---
        if t0 >= next_due["scorecard"]:
            next_due["scorecard"] = t0 + P["scorecard_interval"]
            targets = list(dict.fromkeys(live + recent_done))[:8]
            for mid in targets:
                sc = cl.fetch_scorecard(mid)
                if sc:
                    h = cl.content_hash(sc)
                    if hashes.get(("scard", mid)) != h:
                        append("match_scorecard", [{"match_id": mid, "payload": json.dumps(sc),
                                                    "payload_hash": h, "fetch_ts": cl.now_utc(),
                                                    "processed": False}])
                        hashes[("scard", mid)] = h

        # --- slow loop: schedule, series news, points tables ---
        if t0 >= next_due["slow"]:
            next_due["slow"] = t0 + P["slow_interval"]
            for cat in cl.SCHEDULE_CATEGORIES:
                try:
                    sch = cl.fetch_schedule(cat)
                    if sch:
                        h = cl.content_hash(sch)
                        if hashes.get(("sched", cat)) != h:
                            append("schedule_pages", [{"category": cat, "payload": json.dumps(sch),
                                                       "fetch_ts": cl.now_utc(), "processed": False}])
                            hashes[("sched", cat)] = h
                except Exception as e:
                    cl.log_etl(spark, f"schedule:{cat}", "ERROR", 0, str(e))
            for sid, (sname, slug) in list(series_map.items())[:int(P["series_cap"])]:
                if not slug:
                    continue
                try:
                    nw = cl.fetch_series_news(sid, slug)
                    if nw:
                        append("news_pages", [{"series_id": sid, "series_name": sname,
                                               "payload": json.dumps(nw), "fetch_ts": cl.now_utc(),
                                               "processed": False}])
                except Exception:
                    pass
                try:
                    pt = cl.fetch_points_table(sid, slug)
                    if pt and pt.get("pointsTable"):
                        h = cl.content_hash(pt)
                        if hashes.get(("pt", sid)) != h:
                            append("points_table_pages", [{"series_id": sid, "series_name": sname,
                                                           "payload": json.dumps(pt), "fetch_ts": cl.now_utc(),
                                                           "processed": False}])
                            hashes[("pt", sid)] = h
                except Exception:
                    pass

        # --- hourly: rankings ---
        if t0 >= next_due["rankings"]:
            next_due["rankings"] = t0 + P["rankings_interval"]
            for gender in cl.RANKING_GENDERS:
                for cat in cl.RANKING_CATEGORIES:
                    try:
                        rk = cl.fetch_rankings(gender, cat)
                        if rk:
                            append("rankings_pages", [{"gender": gender, "category": cat,
                                                       "payload": json.dumps(rk), "fetch_ts": cl.now_utc(),
                                                       "processed": False}])
                    except Exception:
                        pass

        # --- transforms: bronze -> silver -> gold, every cycle ---
        stats = cl.process_all_pending(spark)
        cl.refresh_gold(spark, stats)
        cl.update_pipeline_status(spark, "ingest_cycle", sum(v for v in stats.values() if v > 0))
        cl.log_etl(spark, f"cycle:{cycle}", "OK", sum(v for v in stats.values() if v > 0),
                   f"live={len(live)} pending={stats}")

    except Exception as e:
        consecutive_failures += 1
        cl.log_etl(spark, f"cycle:{cycle}", "ERROR", 0, repr(e)[:450])
        print(f"cycle {cycle} ERROR: {e!r}")
        if consecutive_failures >= 5:
            print("5 consecutive failures — backing off 60s")
            time.sleep(60)
            consecutive_failures = 0

    elapsed = time.time() - t0
    if cycle % 10 == 1 or elapsed > P["fast_interval"]:
        print(f"cycle {cycle}: {elapsed:.1f}s, live_pages={list(live_rr)}, stats={stats}")
    time.sleep(max(0.5, P["fast_interval"] - elapsed))